- Creating Catalog,Schema,External Location

In [0]:
def log(msg):
    print(f"[INFO] {msg}")

def warn(msg):
    print(f"[WARN] {msg}")

def error(msg):
    print(f"[ERROR] {msg}")

# 1. Catalog
# -------------------------
try:
    log("Creating catalog dq_project")
    spark.sql("CREATE CATALOG IF NOT EXISTS dq_project")
except Exception as e:
    warn(f"Catalog may already exist or no permission: {str(e)}")

# 2. Schemas
# -------------------------
try:
    log("Creating schemas")
    spark.sql("CREATE SCHEMA IF NOT EXISTS dq_project.bronze")
    spark.sql("CREATE SCHEMA IF NOT EXISTS dq_project.silver")
    spark.sql("CREATE SCHEMA IF NOT EXISTS dq_project.gold")
except Exception as e:
    warn(f"Schemas creation issue: {str(e)}")


# 3. External Location
# -------------------------
try:
    log("Creating external location dq_project_ext_loc")
    spark.sql("""
    CREATE EXTERNAL LOCATION IF NOT EXISTS dq_project_ext_loc
    URL 'abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/'
    WITH (STORAGE CREDENTIAL dq_project_cred)
    COMMENT 'External location for dq_project medallion pipeline'
    """)
    log("External location ready")
except Exception as e:
    warn(f"External location may already exist or credential issue: {str(e)}")

# FINAL STATUS
# -------------------------
log("Setup step completed (safe mode)")

[INFO] Creating catalog dq_project
[INFO] Creating schemas
[INFO] Creating external location dq_project_ext_loc
[INFO] External location ready
[INFO] Setup step completed (safe mode)


- Creating Source  Delta Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dq_project.bronze.source_raw
USING DELTA
LOCATION 'abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze/tables/source_raw_v2';

- Creating target Delta Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dq_project.bronze.target_raw
USING DELTA
LOCATION 'abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze/tables/target_raw_v2';

- Checking Visibility of the tables

In [0]:
%sql
SHOW TABLES IN dq_project.bronze;

database,tableName,isTemporary
bronze,source_raw,false
bronze,target_raw,false
,_sqldf,true


- Logging helper

In [0]:
def log(msg):
    print(f"[INFO] {msg}")

def error(msg):
    print(f"[ERROR] {msg}")

- Loading Source Data

In [0]:
try:
    log("Loading source data into dq_project.bronze.source_raw")

    spark.sql("""
    INSERT OVERWRITE dq_project.bronze.source_raw
    SELECT
      *,
      'source' AS dataset_type,
      current_timestamp() AS ingestion_ts
    FROM read_files(
        'abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze /raw/source',
      format => 'csv',
      header => true,
      inferSchema => true
    )
    """)

    log("Source Bronze load completed successfully")

except Exception as e:
    error(f"Source Bronze load failed: {str(e)}")
    raise

[INFO] Loading source data into dq_project.bronze.source_raw
[INFO] Source Bronze load completed successfully


- Loading Target  Data

In [0]:
try:
    log("Loading target data into dq_project.bronze.target_raw")

    spark.sql("""
    INSERT OVERWRITE dq_project.bronze.target_raw
    SELECT
      *,
      'target' AS dataset_type,
      current_timestamp() AS ingestion_ts
    FROM read_files(
     'abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze /raw/target',
      format => 'csv',
      header => true,
      inferSchema => true
    )
    """)

    log("Target Bronze load completed successfully")

except Exception as e:
    error(f"Target Bronze load failed: {str(e)}")
    raise

[INFO] Loading target data into dq_project.bronze.target_raw
[INFO] Target Bronze load completed successfully


- Validating Loaded Data 

In [0]:
try:
    log("Validating Bronze row counts")

    display(spark.sql("SELECT COUNT(*) AS source_count FROM dq_project.bronze.source_raw"))
    display(spark.sql("SELECT COUNT(*) AS target_count FROM dq_project.bronze.target_raw"))

except Exception as e:
    error(f"Bronze count validation failed: {str(e)}")
    raise

[INFO] Validating Bronze row counts


source_count
2500


target_count
2520


- Validating and Displaying Data 

In [0]:
display(spark.sql("SELECT * FROM dq_project.bronze.source_raw LIMIT 10"))

customer_id,first_name,last_name,gender,city,signup_date,age,annual_income,credit_score,tenure_months,is_active,purchase_count,loyalty_score,risk_band,_rescued_data,dataset_type,ingestion_ts
100001,Diya,Kulkarni,F,Chennai,2020-08-17,40,975535.0,687.0,39,1,15,48.4,Medium,null,source,2026-04-01T19:56:30.35933Z
100002,Vivaan,Mehta,M,Mumbai,2021-12-03,35,1063001.0,685.0,21,0,13,77.7,Medium,null,source,2026-04-01T19:56:30.35933Z
100003,Anaya,Nair,F,Bengaluru,2023-11-28,42,777112.0,611.0,40,1,12,63.2,High,null,source,2026-04-01T19:56:30.35933Z
100004,Aisha,Malhotra,F,Kolkata,2020-05-23,50,715308.0,692.0,82,1,10,89.6,Medium,null,source,2026-04-01T19:56:30.35933Z
100005,Anaya,Mishra,M,null,2020-03-08,34,611267.0,750.0,54,1,16,78.7,Low,null,source,2026-04-01T19:56:30.35933Z
100006,Rahul,Mehta,F,Bengaluru,2022-10-14,34,668410.0,640.0,12,1,12,55.8,Medium,null,source,2026-04-01T19:56:30.35933Z
100007,Isha,Malhotra,M,Ahmedabad,2020-04-07,50,1498801.0,768.0,81,1,17,69.0,Low,null,source,2026-04-01T19:56:30.35933Z
100008,Kabir,Bose,F,Kolkata,2020-01-07,43,1123830.0,737.0,37,1,14,79.4,Low,null,source,2026-04-01T19:56:30.35933Z
100009,Anaya,Kapoor,M,Chennai,2022-03-25,32,552764.0,670.0,38,1,10,70.4,Medium,null,source,2026-04-01T19:56:30.35933Z
100010,Aarav,Saxena,F,Hyderabad,2020-06-25,41,559059.0,702.0,15,1,8,75.7,Medium,null,source,2026-04-01T19:56:30.35933Z


In [0]:
display(spark.sql("SELECT * FROM dq_project.bronze.target_raw LIMIT 10"))

customer_id,first_name,last_name,gender,city,signup_date,age,annual_income,credit_score,tenure_months,is_active,purchase_count,loyalty_score,preferred_channel,_rescued_data,dataset_type,ingestion_ts
100001,Diya,Kulkarni,F,Chennai,2020-08-17,40,1080183.0,689.0,39,1,17,48.3,App,null,target,2026-04-03T06:24:32.629195Z
100002,Vivaan,Mehta,M,Mumbai,2021-12-03,35,1110299.0,674.0,21,0,16,81.0,Email,null,target,2026-04-03T06:24:32.629195Z
100003,Anaya,Nair,F,null,2023-11-28,42,841204.0,600.0,40,1,14,59.7,App,null,target,2026-04-03T06:24:32.629195Z
100004,Aisha,Malhotra,F,Kolkata,2020-05-23,50,754164.0,680.0,82,1,12,93.6,App,null,target,2026-04-03T06:24:32.629195Z
100005,Anaya,Mishra,M,null,2020-03-08,34,663744.0,734.0,54,1,17,78.9,Email,null,target,2026-04-03T06:24:32.629195Z
100006,Rahul,Mehta,F,Bengaluru,2022-10-14,34,737568.0,646.0,12,1,14,58.1,SMS,null,target,2026-04-03T06:24:32.629195Z
100007,Isha,Malhotra,M,Ahmedabad,2020-04-07,50,1597856.0,749.0,81,1,19,73.0,Email,null,target,2026-04-03T06:24:32.629195Z
100008,Kabir,Bose,F,Kolkata,2020-01-07,43,1226489.0,712.0,37,1,13,78.6,SMS,null,target,2026-04-03T06:24:32.629195Z
100009,Anaya,Kapoor,M,Chennai,2022-03-25,32,568089.0,654.0,38,1,11,73.0,SMS,null,target,2026-04-03T06:24:32.629195Z
100010,Aarav,Saxena,F,Hyderabad,2020-06-25,41,583763.0,693.0,15,1,8,73.9,SMS,null,target,2026-04-03T06:24:32.629195Z


- Describing Details 

In [0]:
display(spark.sql("DESCRIBE DETAIL dq_project.bronze.source_raw"))
display(spark.sql("DESCRIBE DETAIL dq_project.bronze.target_raw"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,748f5507-7c89-4072-89f2-8967ea4389f5,dq_project.bronze.source_raw,null,abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze/tables/source_raw_v2,2026-04-01T19:04:48.612Z,2026-04-01T19:56:31Z,List(),List(),1,58007,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,04dca8a1-cdfc-4b25-a0ae-eacacc4c4230,dq_project.bronze.target_raw,null,abfss://external@stacchrishikeshdeshpande.dfs.core.windows.net/dqproject/bronze/tables/target_raw_v2,2026-04-01T19:05:54.331Z,2026-04-01T19:57:18Z,List(),List(),1,58417,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
